In [1]:
import os
os.chdir('../../..')
folder = 'trained_models/cls_embeddings'
os.getcwd()

'/home/trath/Code/nlpvise/src'

In [6]:
from skorch import NeuralNetClassifier

from models.convlstm import ConvLSTM
from models.convnet import ConvNet
from models.bilstm import BiLSTM
from models.mlp import mlp
from models.moe import MoE

In [7]:
def load_model(module, param_path):
    net = NeuralNetClassifier(module=module)
    net.initialize()
    net.load_params(f_params=param_path)

    return net.module_

In [8]:
experts = []
with os.scandir(folder) as entries:
    for entry in entries:
        print(entry.name)
        module = None
        match entry.name:
            case 'BiLSTM_CLS.pkl':
                module = load_model(module=BiLSTM(input_size=768), param_path=entry.path)
            case 'mlp_CLS.pkl':
                module = load_model(module=mlp(input_size=768), param_path=entry.path)
            case 'ConvLSTM_CLS.pkl':
                module = load_model(module=ConvLSTM(input_size=768), param_path=entry.path)
            case 'ConvNet_CLS.pkl':
                module = load_model(module=ConvNet(input_size=768), param_path=entry.path)
        experts.append(module)

ConvNet_CLS.pkl
BiLSTM_CLS.pkl
ConvLSTM_CLS.pkl
mlp_CLS.pkl


In [9]:
experts

[ConvNet(
   (fc1): Linear(in_features=768, out_features=289, bias=True)
   (conv1): Conv2d(1, 19, kernel_size=(3, 3), stride=(1, 1))
   (fc2): Linear(in_features=4275, out_features=19, bias=True)
 ),
 BiLSTM(
   (fc1): Linear(in_features=768, out_features=289, bias=True)
   (lstm): LSTM(289, 300, bidirectional=True)
   (fc2): Linear(in_features=600, out_features=19, bias=True)
 ),
 ConvLSTM(
   (fc1): Linear(in_features=768, out_features=289, bias=True)
   (conv1): Conv2d(1, 19, kernel_size=(3, 3), stride=(1, 1))
   (fc2): Linear(in_features=4275, out_features=289, bias=True)
   (lstm): LSTM(289, 300)
   (fc3): Linear(in_features=300, out_features=19, bias=True)
 ),
 mlp(
   (fc1): Linear(in_features=768, out_features=75, bias=True)
   (fc2): Linear(in_features=75, out_features=19, bias=True)
 )]

In [10]:
for expert in experts:
    for param in expert.parameters():
        param.requires_grad = False

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score

def evaluate_multi_label(y_true, y_probs, threshold=0.6):
    y_pred = (y_probs > threshold).astype(int)
    
    per_class_acc = []
    # per_class_mcc = []

    for i in range(y_true.shape[1]):
        per_class_acc.append(accuracy_score(y_true[:, i], y_pred[:, i]))
        # per_class_mcc.append(matthews_corrcoef(y_true[:, i], y_pred[:, i]))

    results = {
        'accuracy': np.mean(per_class_acc),
        # 'mcc': np.mean(per_class_mcc),
        # 'f1': f1_score(y_true, y_pred, average='weighted'),
        # 'auprc': average_precision_score(y_true, y_probs),
        # 'auroc': roc_auc_score(y_true, y_probs)
    }
    # return results
    return np.mean(per_class_acc)

def train_model_torch(model, X_train, y_train, X_test, y_test,
                      lr=0.001, batch_size=128, epochs=8, device='cpu'):
    accuracies = []
    # Move model to device
    model = model.to(device)

    # Data preparation
    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test = torch.tensor(y_test, dtype=torch.float32).to(device)

    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Loss and optimizer
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Training loop
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X).squeeze()
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f}")

        # Evaluation
        with torch.no_grad():
            model.eval()
            # logits = model(X_test).squeeze()
            # probs = torch.sigmoid(logits)
            # preds = (probs > 0.5).float()
            logits = model(X_test)
            probs = torch.sigmoid(logits).cpu().numpy()
            y_test_np = y_test.cpu().numpy()
            acc = evaluate_multi_label(y_test_np, probs)
            accuracies.append(acc)
            print(f"Accuracy {acc}")

    return model, accuracies

In [13]:
import pickle
with open("data/X.pkl", "rb") as f:
    X = pickle.load(f)

with open("data/y.pkl", "rb") as f:
    y = pickle.load(f)

In [15]:
from sklearn.model_selection import train_test_split

moe_model = MoE(experts=experts, input_dim=768, hidden_dim=128)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'

trained_model, acc = train_model_torch(
    model=moe_model,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    lr=1e-3,
    batch_size=128,
    epochs=8,
    device=device
)

print(f"✅ Mean Accuracy: {np.mean(acc):.4f} ± {np.std(acc):.4f}")

Epoch 1/8 | Loss: 0.4355
Accuracy 0.7832590765493003
Epoch 2/8 | Loss: 0.4354
Accuracy 0.7829979741416025
Epoch 3/8 | Loss: 0.4352
Accuracy 0.7831275994503887
Epoch 4/8 | Loss: 0.4352
Accuracy 0.7830627867959956
Epoch 5/8 | Loss: 0.4352
Accuracy 0.7829331614872096
Epoch 6/8 | Loss: 0.4351
Accuracy 0.7830405653144896
Epoch 7/8 | Loss: 0.4350
Accuracy 0.7826665037091356
Epoch 8/8 | Loss: 0.4350
Accuracy 0.7829257543267076
✅ Mean Accuracy: 0.7830 ± 0.0002
